# Phase 2

1. Loads the PhiUSIIL phishing URL dataset and has a quick look at it (Task 1).
2. Does the bare minimum of cleaning so the eLCS code can read the data.
3. Trains the *original, unmodified* eLCS with its default settings and checks how well it does (Task 2).

**S1** uses all 50 numeric features, **S2** removes 4 "derived" features that might leak the answer.
Phishing is treated as the "positive" class, so *recall* = how many phishing sites we catch.

In [1]:
import sys, time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

from skeLCS import eLCS                              # the original eLCS, nothing inside it is changed

CSV = "PhiUSIIL_Phishing_URL_Dataset.csv"            # the dataset file
SEED = 0                                             # makes the split and the eLCS run repeatable

## 1. Looking at the data (Task 1)

In [2]:
df = pd.read_csv(CSV)
print("rows, columns:", df.shape)
print()
print("label counts (1 = legitimate, 0 = phishing):")
print(df["label"].value_counts())
print()
print("missing values:", int(df.isna().sum().sum()))
print("rows that repeat an earlier URL:", int(df["URL"].duplicated().sum()))
df.head(3)

rows, columns: (235795, 56)

label counts (1 = legitimate, 0 = phishing):
label
1    134850
0    100945
Name: count, dtype: int64

missing values: 0
rows that repeat an earlier URL: 425


,FILENAME,URL,URLLength,Domain,DomainLength,IsDomainIP,TLD,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,...,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef,label
0,521848.txt,https://www.southbankmosaics.com,31,www.southbankmosaics.com,24,0,com,100.0,1.000000,0.522907,...,0,0,1,34,20,28,119,0,124,1
1,31372.txt,https://www.uni-mainz.de,23,www.uni-mainz.de,16,0,de,100.0,0.666667,0.032650,...,0,0,1,50,9,8,39,0,217,1
2,597387.txt,https://www.voicefmradio.co.uk,29,www.voicefmradio.co.uk,22,0,uk,100.0,0.866667,0.028555,...,0,0,1,10,2,7,42,2,5,1


**What we see (Phase I recap)**
- About 236k pages, 56 columns: 50 numeric features, 5 text/ID columns (`FILENAME`, `URL`, `Domain`, `TLD`, `Title`) and the `label`.
- Nothing is missing. A few hundred rows repeat a URL we've already seen.
- Classes are a bit lopsided (about 57% legitimate, 43% phishing), but not badly.

## 2. Minimal cleaning
eLCS only understands numbers, so the 5 text columns have to go. Everything else stays untouched
(no scaling, no outlier removal) because Task 2 is about the original eLCS on raw data.

The one thing we do is remove repeated URLs, so the same page can't end up in both the training and test sets.

In [3]:
TEXT_COLS = ["FILENAME", "URL", "Domain", "TLD", "Title"]
# 4 features that are calculated from dataset-wide numbers so they could leak the answer.
# (e.g. URLSimilarityIndex is exactly 100 for every legitimate URL.)
DERIVED = ["URLSimilarityIndex", "TLDLegitimateProb", "URLCharProb", "CharContinuationRate"]

df = df.drop_duplicates(subset="URL")                  # keep the first copy of each URL
y = (df["label"] == 0).astype(int).values              # 1 = phishing (the "positive" class), 0 = legitimate
X_all = df.drop(columns=TEXT_COLS + ["label"])         # numeric features only

print("rows after removing repeated URLs:", len(df))
print("numeric features:", X_all.shape[1])

rows after removing repeated URLs: 235370
numeric features: 50
